# Computing metrics (Potential skill, Conditional Bias, Unconditional Bias, Skill Score, Tercile Hit Rate) for seasonal data.

This notebook walks through how to compute the seasonal metrics for each model, used in the seasonal heatmap plots.

- We compute all the metrics by model, region, and season, including for MME. The data generated here is later taken to choose the models in SMME. (selection process will not include using MME's metrics, MME metrics are computed here for convienience) After running the selection process for SMME, this file will have to be run again to generate the metrics for SMME.

- The metrics (potential skill, AN hitrate, etc.) must be saved to some save path for the SMME data generation file to work. This path is defined under the import statement.

In [ ]:
# import necessary packages
import numpy as np
import pandas as pd
import os

In [ ]:
'''Change all these paths to your paths
'''

# path to seasonal csv data folder (change to your path)
data_dir = 'data/seasonal/csv'

# metrics data save path (change to your path)
metrics_save_path = 'data/csv/metrics'

# Compute Seasonal Potential Skill, Unconditional Bias, Conditional Bias, and Skill Score

In [ ]:
# Initialize an empty dictionary to store DataFrames
dfs_dict = {}

# Loop over all files in the directory
for filename in os.listdir(data_dir):
    if filename.endswith('_seasonal.csv'):
        # Full path to the file
        file_path = os.path.join(data_dir, filename)
        # Use filename without extension as key
        file_key = os.path.splitext(filename)[0]
        # Read the CSV file into a DataFrame
        df = pd.read_csv(file_path, sep=',', header=0, index_col=False)
        # Store in dictionary
        dfs_dict[file_key] = df

        # drop unnamed 0 column
        dfs_dict[file_key] = dfs_dict[file_key].drop(columns=['Unnamed: 0'], errors='ignore')

# Concatenate all DataFrames
final_df = pd.concat(dfs_dict.values()).reset_index(drop=True)

# Drop unnecessary columns
final_df = final_df.drop(columns=['year_of_prediction', 'realization_year', 'index'], errors='ignore')

# Normalize precipitation values
final_df['precip'] = final_df['precip'] / 30


In [ ]:
# compute potential skill, unconditional bias, skill score, and conditional bias
# Calculating all the statistics for each region and model
# groupby the region, model, season and month_of_prediction and calculate the corr between the predicted and actual precipitation for future use
corr = final_df.groupby(['region', 'model', 'season', 'month_of_prediction', 'lead_time'])[['predicted_precip', 'precip']].corr(method = 'spearman').drop(['precip'], axis = 1).reset_index()
corr = corr.drop(corr.index[::2]).drop(columns = ['level_5'])
corr = corr.rename(columns = {'predicted_precip': 'corr'})

# Calculate mean and standard deviation
stat = final_df.groupby(['region', 'model', 'season', 'month_of_prediction', 'lead_time']).agg(['mean', 'std']).reset_index()
stat.columns = ['region', 'model', 'season', 'month_of_prediction', 'lead_time', 'pred_mean', 'pred_std', 'actual_mean', 'actual_std']

# Merging stat and spatial_means_corr to get 1 df with all values
stat_clean = stat.merge(corr, left_on=['region', 'model', 'season', 'month_of_prediction', 'lead_time'],
                        right_on=['region', 'model', 'season', 'month_of_prediction', 'lead_time'], how='left').dropna()

# Calculating metrics
stat_clean['potential_skill'] = np.square(stat_clean['corr'])
stat_clean['conditional_bias'] = np.square(stat_clean['corr'] - (stat_clean['pred_std'] / stat_clean['actual_std']))
stat_clean['unconditional_bias'] = np.square((stat_clean['pred_mean'] - stat_clean['actual_mean']) / stat_clean['actual_std'])
stat_clean['skill_score'] = stat_clean['potential_skill'] - stat_clean['conditional_bias'] - stat_clean['unconditional_bias']

# Save the skill metrics DataFrame to a CSV file
stat_clean.to_csv(os.path.join(metrics_save_path, 'skill_metrics.csv'), index=False)

# Compute Seasonal Tercile Hit Rate

In [ ]:
# compute tercile agreement rate

# define tercile cutoff helper functions

# get the tercile cutoffs of a dataframe
def get_tercile_cutoffs(df):
    return df.quantile([0.33, 0.66])

# Assign the Tercile Category for both predicted_precip and precip
def assign_tercile_category(value, lower_cutoff, upper_cutoff):
    if value <= lower_cutoff:
        return 'Low'
    elif value <= upper_cutoff:
        return 'Medium'
    else:
        return 'High'

def compute_tercile_seasonal_df(seasonal_file_path):
    """Takes a file path to seasonal csv file and computes tercile
    agreement for each month of prediction and season.

    Requires helper functions get_tercile_cutoffs and assign_tercile_category,
    which are defined above.

    Arguments
    ---------
    seasonal_file_path (str): file path to seasonal csv file
    make sure that the path provided is a path to the actual file:
    /data/csv/{some_region}_{some_model}_merged_seasonal.csv

    The csv file has the following columns:
    'date_of_prediction' - model's date of prediction as YYYY-MM-DD
    'season' - season that the model is predicting, i.e. JJA, JAS, SON, DJF, MAM
    'predicted_precip' - predicted precipitation from model
    'precip' - actual precipitation from CHIRPS
    'model' - model name, i.e. NCEP, CFSv2, ECMWF, GFS
    'region' - region name, i.e. eastern_east_africa, west_africa, etc.

    Returns
    -------
    dataframe with tercile agreement for each month of prediction and season

    The resulting dataframe has the following columns:
    'year_of_prediction' - year of prediction as YYYY
    'season' - season that the model is predicting, i.e. JJA, JAS, SON, DJF, MAM
    'month_of_prediction' - month of prediction as MM
    'predicted_precip' - predicted precipitation from model
    'precip' - actual precipitation from CHIRPS
    f'{model}_tercile_class' - tercile class (low/medium/high) of predicted precipitation from model,
                                column name changes dynamically
    'chirps_tercile_class' - tercile class of actual precipitation from CHIRPS
    'agreement' - 1 if terciles agree, 0 otherwise
    'model' - model name, i.e. NCEP, CFSv2, ECMWF, GFS
    'region' - region name, i.e. eastern_east_africa, west_africa, etc.
    """

    # open seasonal file
    df = pd.read_csv(seasonal_file_path, index_col=False)

    # Iterate over every unique combination of month_of_prediction and season over all years
    for (month, season), group in df.groupby(['month_of_prediction', 'season']):
        model = df['model'].iloc[0]
        region = df['region'].iloc[0]

        cutoffs = get_tercile_cutoffs(group[['predicted_precip', 'precip']])

        # extract cutoffs for predicted precip and precip
        lower_cutoff_predicted = cutoffs.loc[0.33, 'predicted_precip']
        upper_cutoff_predicted = cutoffs.loc[0.66, 'predicted_precip']

        lower_cutoff_precip = cutoffs.loc[0.33, 'precip']
        upper_cutoff_precip = cutoffs.loc[0.66, 'precip']

        # assign tercile categories for predicted_precip and precip
        group[f'{model}_tercile_class'] = group['predicted_precip'].apply(assign_tercile_category, args=(lower_cutoff_predicted, upper_cutoff_predicted))
        group['chirps_tercile_class'] = group['precip'].apply(assign_tercile_category, args=(lower_cutoff_precip, upper_cutoff_precip))

        # calculate agreement, 1 if terciles agree, 0 otherwise
        group['agreement'] = (group[f'{model}_tercile_class'] == group['chirps_tercile_class']).astype(int)

        # Update the DataFrame with the new columns
        df.loc[group.index, f'{model}_tercile_class'] = group[f'{model}_tercile_class']
        df.loc[group.index, 'chirps_tercile_class'] = group['chirps_tercile_class']
        df.loc[group.index, 'agreement'] = group['agreement']
        df.loc[group.index, 'model'] = model
        df.loc[group.index, 'region'] = region

    # Select the desired columns for the final DataFrame
    final_columns = ['year_of_prediction', 'season', 'month_of_prediction', 'predicted_precip', 'precip', f'{model}_tercile_class', 'chirps_tercile_class', 'agreement', 'model', 'region']
    final_df = df[final_columns]

    return final_df


In [ ]:
# Initialize an empty dictionary to store DataFrames
dfs_dict = {}

# Loop over all files in the directory
for filename in os.listdir(data_dir):
    if filename.endswith('_seasonal.csv'):
            # Generate the DataFrame
        df = compute_tercile_seasonal_df(os.path.join(data_dir, filename))

        # Store the DataFrame in the dictionary with the file name as key
        dfs_dict[filename] = df

    # Concatenate all DataFrames in the dictionary into one DataFrame
    tercile_df = pd.concat(dfs_dict.values(), ignore_index=True)


In [ ]:
# compute AN, BN, and N tercile agreement rates by model and region

region_model_df_BN = tercile_df.query('chirps_tercile_class == "Low"')

# compute agreement rates
region_model_df_BN = region_model_df_BN.groupby(['month_of_prediction', 'season', 'model', 'region'])[['agreement']].mean().reset_index()

# subset by chirps tercile class = medium
region_model_df_N = tercile_df.query('chirps_tercile_class == "Medium"')

# compute agreement rates
region_model_df_N = region_model_df_N.groupby(['month_of_prediction', 'season', 'model', 'region'])[['agreement']].mean().reset_index()

region_model_df_AN = tercile_df.query('chirps_tercile_class == "High"')

region_model_df_AN = region_model_df_AN.groupby(['month_of_prediction', 'season', 'model', 'region'])[['agreement']].mean().reset_index()

# save AN, BN, and N metrics as csv
region_model_df_BN.to_csv(os.path.join(metrics_save_path, 'BN_hitrate_metrics.csv'), index=False)

region_model_df_N.to_csv(os.path.join(metrics_save_path, 'N_hitrate_metrics.csv'), index=False)

region_model_df_AN.to_csv(os.path.join(metrics_save_path, 'AN_hitrate_metrics.csv'), index=False)

# Compute Overall Hitrate

This metric is overall hitrate, effectively combining AN, BN, and N into one robust metric for model skill analysis.

In [15]:
def compute_overall_tercile_seasonal_df(seasonal_file_path):

    df = pd.read_csv(seasonal_file_path, index_col=False)

    # Prepare empty list to collect processed rows
    processed_rows = []

    # Iterate over every unique combination of model and season
    for (model, season), group in df.groupby(['model', 'season']):
        for (month, _), subgroup in group.groupby(['month_of_prediction', 'season']):
            # Compute tercile cutoffs for this subgroup
            cutoffs = get_tercile_cutoffs(subgroup[['predicted_precip', 'precip']])
            lower_cutoff_predicted = cutoffs.loc[0.33, 'predicted_precip']
            upper_cutoff_predicted = cutoffs.loc[0.66, 'predicted_precip']
            lower_cutoff_precip = cutoffs.loc[0.33, 'precip']
            upper_cutoff_precip = cutoffs.loc[0.66, 'precip']

            # Assign tercile classes
            subgroup[f'{model}_tercile_class'] = subgroup['predicted_precip'].apply(
                assign_tercile_category, args=(lower_cutoff_predicted, upper_cutoff_predicted))
            subgroup['chirps_tercile_class'] = subgroup['precip'].apply(
                assign_tercile_category, args=(lower_cutoff_precip, upper_cutoff_precip))

            # Calculate agreement
            subgroup['agreement'] = (
                subgroup[f'{model}_tercile_class'] == subgroup['chirps_tercile_class']
            ).astype(int)

            processed_rows.append(subgroup)

    # Concatenate all subgroups with tercile and agreement columns
    all_data = pd.concat(processed_rows, ignore_index=True)

    # Compute hit rate per (model, season, region, month of prediction)
    summary = (
        all_data.groupby(['model', 'month_of_prediction', 'season', 'region'])
        .agg(
            total_predictions=('agreement', 'count'),
            correct_predictions=('agreement', 'sum')
        )
        .reset_index()
    )
    summary['hit_rate'] = summary['correct_predictions'] / summary['total_predictions']

    return summary

In [17]:
# Initialize an empty dictionary to store DataFrames
dfs_dict = {}

# List all files in the directory
list_of_files = [
    os.path.join(metrics_save_path, f)
    for f in os.listdir(metrics_save_path)
    if f.endswith('_merged_seasonal.csv')
]

# Loop over all files
for f in list_of_files:
    # Generate the DataFrame
    df = compute_overall_tercile_seasonal_df(f)

    # Store the DataFrame in the dictionary with the file path as key
    dfs_dict[f] = df

# Concatenate all DataFrames in the dictionary into one DataFrame
final_df = pd.concat(dfs_dict.values(), ignore_index=True)

# Save to CSV
final_df.to_csv(os.path.join(metrics_save_path, 'overall_seasonal_hitrate.csv'), index=False)